# 11 — Chest X-ray Segmentation U-Net : controle qualite avant experimentation

**Auteur :** Yann Smatti  
**Objectif :** verifier que les paires image/masque thoraciques sont exploitables avant entrainement et evaluation

Sur ce sujet, la plus grande source d'erreur vient souvent du dataset lui-meme : masque mal aligne, manifest incomplet, ou radio mal normalisee. Ce notebook sert donc d'etape de controle qualite avant toute conclusion sur le modele.

## Ce que je cherche a confirmer

- la structure de donnees est exploitable sans correction manuelle massive ;
- les masques recouvrent bien la zone attendue ;
- les premiers overlays ne revelent pas de decalage evident ;
- le flux de preparation est assez sain pour justifier un vrai cycle d'entrainement.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from src.utils.config import load_config
from src.segmentation.datasets.manifest import build_manifest

cfg = load_config('configs/chest_xray_segmentation.yaml')
raw_dir = Path(cfg['raw_dataset_dir'])
manifest_path = Path(cfg['manifest_path'])
print(cfg)
print('raw_dir exists =', raw_dir.exists())


In [ ]:
if raw_dir.exists():
    if (not manifest_path.exists()) or manifest_path.stat().st_size == 0:
        df = build_manifest(raw_dir, manifest_path, cfg['class_names'])
        print('Manifest régénéré, lignes =', len(df))
    else:
        df = pd.read_csv(manifest_path)
        print('Manifest chargé, lignes =', len(df))
else:
    df = pd.DataFrame()
    print('Dataset absent pour le moment')

df.head()


In [ ]:
if not df.empty:
    display(df['label'].value_counts(dropna=False).to_frame('count'))
    sample = df.iloc[0]
    image = np.asarray(Image.open(sample['image_path']).convert('RGB'))
    mask = np.asarray(Image.open(sample['mask_path']).convert('L')) / 255.0

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(image)
    axes[0].set_title('Image')
    axes[0].axis('off')

    axes[1].imshow(mask, cmap='gray')
    axes[1].set_title('Mask')
    axes[1].axis('off')

    axes[2].imshow(image)
    axes[2].imshow(mask, cmap='Reds', alpha=0.35)
    axes[2].set_title('Overlay')
    axes[2].axis('off')
    plt.tight_layout()


## Point cle

Pour la radio thoracique, ce notebook doit etre considere comme un garde-fou avant apprentissage. Une segmentation apparente peut etre trompeuse si les donnees sont mal alignees ou si les masques ont ete generes selon une logique heterogene.

## Conditions minimales avant la suite

- verifier plusieurs overlays, pas uniquement le premier exemple ;
- mesurer la proportion de masques quasi vides ou anormalement larges ;
- figer la procedure de split avant l'entrainement ;
- conserver ce notebook comme trace du controle qualite initial.

Si ces controles ne sont pas satisfaisants, le probleme est d'abord data-centric et non modele-centric.